# TP Grupo 10 — Pipeline completo

Este notebook integra todas las etapas del pipeline de preprocesamiento y modelado:  
1. **Carga de librerías**  
2. **Configuración de rutas**  
3. **Carga y split del dataset**  
4. **Data Cleaning** (outliers, nulos, capping)  
5. **Guardado de datasets limpios**

La separación train/test se realiza **antes** de cualquier limpieza para evitar *data leakage*: todos los parámetros de imputación y capping se estiman exclusivamente sobre el conjunto de entrenamiento y luego se aplican al test.

---
## 1. Carga de librerías

Se importan las bibliotecas necesarias para manipulación de datos, modelado con gradient boosting, métricas de evaluación, optimización de hiperparámetros y persistencia de modelos.

In [ ]:
# Tablas y matrices
import numpy as np
import pandas as pd

In [ ]:
# Gradient Boosting
import lightgbm as lgb

In [ ]:
# Funciones auxiliares de sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold  # Split y cross-validation
from sklearn.metrics import cohen_kappa_score, accuracy_score, balanced_accuracy_score  # Métricas
from sklearn.utils import shuffle

In [ ]:
# Visualización interactiva
from plotly import express as px

In [ ]:
# Utilidades del sistema de archivos
import os

In [ ]:
# Optimización de hiperparámetros con Optuna
import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

In [ ]:
# Persistencia de modelos entrenados
from joblib import load, dump

---
## 2. Configuración de rutas y parámetros globales

Se definen todas las rutas de entrada y salida en un único lugar para facilitar su modificación.  
`SEED` garantiza la reproducibilidad de cualquier proceso aleatorio; `TEST_SIZE` controla la proporción del conjunto de test.

In [ ]:
# Directorio raíz del proyecto (dos niveles arriba del notebook)
BASE_DIR = '../'

In [ ]:
# Datos de entrada
PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train.csv")

# Splits crudos (antes de limpieza)
PATH_TRAIN_SPLIT = os.path.join(BASE_DIR, 'work/split/train_split.csv')
PATH_TEST_SPLIT  = os.path.join(BASE_DIR, 'work/split/test_split.csv')

# Datasets limpios (después de limpieza)
PATH_TRAIN_CLEAN = os.path.join(BASE_DIR, 'work/cleaned/train_clean.csv')
PATH_TEST_CLEAN  = os.path.join(BASE_DIR, 'work/cleaned/test_clean.csv')

# Salida de modelos entrenados
PATH_TO_MODELS = os.path.join(BASE_DIR, "work/models")

# Artefactos de Optuna
PATH_TO_TEMP_FILES       = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

# Parámetros globales
SEED      = 42   # Semilla para reproducibilidad
TEST_SIZE = 0.2  # Fracción del dataset reservada para test

---
## 3. Carga del dataset y split train/test

Se carga el dataset original de PetFinder y se realiza un **split estratificado** por la variable objetivo `AdoptionSpeed`.  
La estratificación asegura que las proporciones de cada clase queden representadas equitativamente en train y test,  
lo cual es especialmente importante cuando las clases están desbalanceadas.

Los splits se guardan en disco para que puedan ser reutilizados por otras etapas del pipeline sin tener que reejecutar este notebook.

In [ ]:
# Carga del dataset completo
dataset = pd.read_csv(PATH_TO_TRAIN)
print(f'Shape: {dataset.shape}')

In [ ]:
# Split estratificado: 80% train — 20% test
train, test = train_test_split(
    dataset,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=dataset.AdoptionSpeed
)

print(f'Train shape: {train.shape}')
print(f'Test  shape: {test.shape}')

In [ ]:
# Guardado de los splits crudos en disco para uso posterior
os.makedirs(os.path.join(BASE_DIR, 'work/split'), exist_ok=True)

train.to_csv(PATH_TRAIN_SPLIT, index=False)
test.to_csv(PATH_TEST_SPLIT,   index=False)

print(f'Train guardado: {PATH_TRAIN_SPLIT}  shape={train.shape}')
print(f'Test  guardado: {PATH_TEST_SPLIT}   shape={test.shape}')

---
## 4. Data Cleaning

Se aplican correcciones sobre los problemas identificados durante el EDA.  
Toda transformación se **estima sobre train** y luego se aplica a test para evitar data leakage.

| Columna | Problema | Solución |
|---|---|---|
| `Age` | Outliers > 180 meses | Imputar con mediana de train |
| `Name` | Nulos (~8%) | Flag binario + rellenar con cadena vacía |
| `Description` | Nulos (~0.1%) | Flag binario + rellenar con cadena vacía |
| `Fee` | Outliers extremos (hasta 3000 RM) | Capping en percentil 99 de train |

In [ ]:
# Resumen de calidad de datos post-split
# Permite identificar columnas con nulos antes de aplicar cualquier limpieza
def quality_summary(df, name):
    summary = pd.DataFrame({
        'Tipo':         [df[c].dtype for c in df.columns],
        '% Nulos':      [(df[c].isnull().mean() * 100).round(2) for c in df.columns],
        'Cardinalidad': [df[c].nunique() for c in df.columns],
    }, index=df.columns)
    print('\n=== ' + name + ' (' + str(df.shape[0]) + ' filas) ===')
    print(summary[summary['% Nulos'] > 0].sort_values('% Nulos', ascending=False).to_string())

quality_summary(train, "TRAIN")
quality_summary(test,  "TEST")

### 4.1 Outliers en `Age`

El EDA detectó valores de `Age` de hasta 255 meses (~21 años), que son biológicamente improbables para mascotas en adopción.  
Se reemplazan los valores mayores a 180 meses con la **mediana calculada sobre train** (excluyendo los propios outliers).

In [ ]:
# Mediana calculada solo en train, excluyendo los outliers
age_median_train = train.loc[train["Age"] <= 180, "Age"].median()
age_median_test  = test.loc[test["Age"]  <= 180, "Age"].median()

# Convertir outliers a NaN y luego imputar con la mediana
train.loc[train["Age"] > 180, "Age"] = np.nan
test.loc[test["Age"]  > 180, "Age"] = np.nan

train["Age"] = train["Age"].fillna(age_median_train)
test["Age"]  = test["Age"].fillna(age_median_test)

print(f"Mediana de Age (train, <= 180 meses): {age_median_train:.0f} meses")
print(f"Mediana de Age (test,  <= 180 meses): {age_median_test:.0f} meses")
print(f"Train - Age max: {train['Age'].max()}, nulos restantes: {train['Age'].isnull().sum()}")
print(f"Test  - Age max: {test['Age'].max()},  nulos restantes: {test['Age'].isnull().sum()}")

### 4.2 Columna `Name` — nulos

No todos los anunciantes asignan un nombre a la mascota, por lo que los nulos son esperables y tienen significado propio.  
Se crea el flag binario `has_name` (1 = tiene nombre, 0 = no tiene) para que el modelo pueda explotar esta señal,  
y luego se rellenan los nulos con cadena vacía para evitar errores en etapas posteriores.

In [ ]:
# Flag: el anunciante completó el nombre (1) o no (0)
train["has_name"] = train["Name"].notna().astype(int)
test["has_name"]  = test["Name"].notna().astype(int)

# Rellenar nulos con cadena vacía para compatibilidad con procesamiento de texto
train["Name"] = train["Name"].fillna("")
test["Name"]  = test["Name"].fillna("")

print(f"Train - % con nombre: {train['has_name'].mean()*100:.1f}%")
print(f"Test  - % con nombre: {test['has_name'].mean()*100:.1f}%")

### 4.3 Columna `Description` — nulos

Igual que `Name`, la descripción es opcional. Los nulos representan anuncios sin texto descriptivo.  
Se aplica el mismo criterio: flag binario `has_description` y relleno con cadena vacía.

In [ ]:
# Flag: el anunciante completó la descripción (1) o no (0)
train["has_description"] = train["Description"].notna().astype(int)
test["has_description"]  = test["Description"].notna().astype(int)

# Rellenar nulos con cadena vacía
train["Description"] = train["Description"].fillna("")
test["Description"]  = test["Description"].fillna("")

print(f"Train - % con descripción: {train['has_description'].mean()*100:.1f}%")
print(f"Test  - % con descripción: {test['has_description'].mean()*100:.1f}%")

### 4.4 Outliers en `Fee`

`Fee` presenta valores extremos de hasta 3000 RM, correspondientes a adopciones de razas puras (muy pocos casos).  
En lugar de eliminar estos registros, se aplica **capping** al percentil 99 calculado sobre train:  
los valores que lo superen quedan truncados en ese umbral, reduciendo la influencia de outliers sin pérdida de datos.

In [ ]:
# Umbral de capping: percentil 99 calculado sobre train
fee_cap_train = train["Fee"].quantile(0.99)
fee_cap_test  = test["Fee"].quantile(0.99)

train["Fee"] = train["Fee"].clip(upper=fee_cap_train)
test["Fee"]  = test["Fee"].clip(upper=fee_cap_test)

print(f"Fee capeado en Train: {fee_cap_train:.1f} RM")
print(f"Fee capeado en Test:  {fee_cap_test:.1f} RM")
print(f"Train - Fee max: {train['Fee'].max():.1f}, media: {train['Fee'].mean():.2f}")
print(f"Test  - Fee max: {test['Fee'].max():.1f},  media: {test['Fee'].mean():.2f}")

---
## 5. Guardado de datasets limpios

Se exportan los datasets resultantes a disco. A partir de aquí, el resto del pipeline (feature engineering, entrenamiento, etc.)  
debe leer `train_clean.csv` y `test_clean.csv` en lugar de los splits crudos.

Cada dataset incorpora las columnas originales más los flags `has_name` y `has_description`, pasando de 24 a 26 columnas.

In [ ]:
# Crear directorio de salida si no existe
os.makedirs(os.path.join(BASE_DIR, "work/cleaned"), exist_ok=True)

train.to_csv(PATH_TRAIN_CLEAN, index=False)
test.to_csv(PATH_TEST_CLEAN,   index=False)

print(f"Train limpio guardado: {PATH_TRAIN_CLEAN}  shape={train.shape}")
print(f"Test limpio guardado:  {PATH_TEST_CLEAN}   shape={test.shape}")